# tuning de hiperparametros

grid search + cross validation para encontrar la mejor config del GA

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from omnievo import (
    DataGenerator,
    GeneticOptimizer,
    grid_search,
    cross_validate,
    sensitivity_analysis,
    analyze_convergence,
)

In [ ]:
# datos
gen = DataGenerator(n_users=2000, random_state=42)
df = gen.generate()
channels = gen.get_channel_names()

X = df[channels].values
y = df['LTV_real'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(f"train: {len(X_train)}, test: {len(X_test)}")

## grid search

In [ ]:
# esto tarda un rato...
param_grid = {
    "population_size": [30, 50, 100],
    "generations": [50, 100],
    "cxpb": [0.6, 0.8],
    "mutpb": [0.1, 0.2],
}

gs = grid_search(X_train, y_train, X_test, y_test, param_grid=param_grid, n_runs=1)

In [ ]:
print(f"mejor config: {gs['best_params']}")
print(f"rmse: {gs['best_result']['rmse_test']:.4f}")
gs['results_df'].head(10)

## sensibilidad

In [ ]:
base = {"population_size": 50, "generations": 100, "cxpb": 0.7, "mutpb": 0.15}

# poblacion
sens_pop = sensitivity_analysis(
    X_train, y_train, X_test, y_test,
    base_params=base,
    param_name="population_size",
    param_values=[20, 50, 100, 150],
    n_runs=2,
)

In [ ]:
plt.figure(figsize=(8, 4))
plt.errorbar(sens_pop['population_size'], sens_pop['rmse_mean'], 
             yerr=sens_pop['rmse_std'], marker='o', capsize=5)
plt.xlabel('poblacion')
plt.ylabel('rmse')
plt.title('sensibilidad a tamaño de poblacion')
plt.grid(alpha=0.3)
plt.show()

## cross validation

In [ ]:
cv = cross_validate(
    X, y,
    k_folds=5,
    optimizer_params=gs['best_params'],
)

print(f"rmse: {cv['rmse_mean']:.4f} +/- {cv['rmse_std']:.4f}")
print(f"pearson: {cv['pearson_mean']:.4f} +/- {cv['pearson_std']:.4f}")

In [ ]:
# rmse por fold
folds = pd.DataFrame(cv['folds'])
plt.bar(folds['fold'], folds['rmse'])
plt.axhline(cv['rmse_mean'], color='r', ls='--')
plt.xlabel('fold')
plt.ylabel('rmse')
plt.show()

## convergencia

In [ ]:
opt = GeneticOptimizer(**gs['best_params'], random_state=42, verbose=False)
result = opt.fit(X_train, y_train)

conv = analyze_convergence(result)
print(f"mejora: {conv['improvement_pct']:.1f}%")
print(f"converge en gen: {conv['convergence_gen']}")

In [ ]:
# pesos finales
print("\npesos:")
for ch, w in sorted(zip(channels, result['best_weights']), key=lambda x: -x[1]):
    bar = '#' * int(w * 30)
    print(f"  {ch:20s} {w:.3f} {bar}")